# Elastic Reverse Time Adjoint Migration (E-RTAM) using a compact formulation 

In this tutorial, we perform an E-RTAM using compact forward and adjoint modeling equations. These equations are based on the stiffness matrix (${\bf C}$) and the ${\bf D}$ and ${\bf S}$ matrices representing the differential operators.

## Forward modeling

The propagation of seismic waves in heterogeneous, isotropic, elastic earth media can be expressed by the elastodynamic equations:

\begin{equation}
\left\{\begin{array}{ll} 
\dfrac{\partial \sigma_{xx}}{\partial t} - (\lambda+2\mu)\dfrac{\partial v_{x}}{\partial x}-\lambda\dfrac{\partial v_{z}}{\partial z}=f_{\sigma_{xx}},\\
\dfrac{\partial\sigma_{zz}}{\partial t}-(\lambda+2\mu)\dfrac{\partial v_{z}}{\partial z} -\lambda\dfrac{\partial v_{x}}{\partial x}=f_{\sigma_{zz}},\\
\dfrac{\partial \sigma_{zx}}{\partial t} -\mu\Big(\dfrac{\partial v_{x}}{\partial z} + \dfrac{\partial v_{z}}{\partial x}\Big) = f_{\sigma_{zx}},\\
\rho\dfrac{\partial v_x}{\partial t} - \Big(\dfrac{\partial \sigma_{xx}}{\partial x} + \dfrac{\partial \sigma_{xz}}{\partial z} )=0,\\
\rho\dfrac{\partial v_z}{\partial t} - \Big(\dfrac{\partial \sigma_{zx}}{\partial x}+\dfrac{\partial \sigma_{zz}}{\partial z}\Big)=0 .
\end{array}\right.
\end{equation}


where $\vec{v}=(v_x,~v_z)$ are the horizontal and vertical particle velocity fields, $\sigma=(\sigma_{xx},~\sigma_{zz},~\sigma_{xz}$) are the stress fields, $f=$($f_{\sigma_{xx}},~f_{\sigma_{xx}})$
are the source terms, $\rho$ is density, $\lambda$ and $\mu$ are the Lame parameters. The elastic wave equation given described above can be organized using a compact formulation as presented by Chen and Sacchi (2020):

\begin{equation}
     \left\{\begin{array}{l}\rho \dfrac{\partial \vec{v}}{\partial t}-{\bf D} \sigma=0,\\ 
     \dfrac{\partial \sigma}{\partial t}-{\bf C D}^{T} \vec{v}=f_\sigma, \end{array}\right.
\end{equation}

being ${\bf C}$ the isotropic elastic tensor in Voigt notation, ${\bf D}$ is a collection of spatial differential operators, defined as

\begin{equation}
\begin{split}
   {\bf C}=\left(\begin{array}{ccc}
   \lambda+2 \mu & \lambda & 0 \\
   \lambda & \lambda+2 \mu & 0 \\ 
   0 & 0 & \mu \end{array}\right)~~\text{e}~~
{\bf D}=\left(\begin{array}{ccc}  
\dfrac{\partial}{\partial x}& 0 &\dfrac{\partial}{\partial z} \\
0 &  \dfrac{\partial}{\partial z}&\dfrac{\partial}{\partial x}
\end{array}\right)  .
\end{split}
\end{equation}

In our compact formulation, the stress tensor $(\sigma)$ is represented is written in vectorial form:

\begin{equation}
\sigma=\left(\begin{array}{c}
 \sigma_{xx}  \\ 
 \sigma_{zz}  \\ 
 \sigma_{zx}
\end{array}\right)
\end{equation}

The conversion from matrix to vector form of the stress tensor is done using the vec() function. An example of this application is $\sigma$=vec($\sigma$).

In [ ]:
# ERRO NA CRIAÇÃO DO MODELO: NÃO DEVE PASSAR PARÂMETRO "B", MAS SIM "RHO"

In [ ]:
from examples.seismic.source import RickerSource, TimeAxis
from examples.seismic import setup_geometry, PointSource, Receiver
from examples.seismic import SeismicModel
from examples.seismic.stiffness.model import ISOSeismicModel
from examples.seismic.stiffness.utils import C_Matrix, D, S, vec
from devito import (Eq, Operator, VectorTimeFunction, TensorTimeFunction,
                    VectorFunction, solve)
from examples.seismic.stiffness import iso_elastic_setup
import numpy as np
import matplotlib.pyplot as plt
from devito import configuration, norm
configuration['log-level'] = 'WARNING'

In [ ]:
nx = 200
nz = 120

shape = (nx, nz)
spacing = (10., 10.)
dx, dz = spacing
origin = (0., 0.)
nlayers = 3
nbl = 50
space_order = 8
dtype = np.float32

# Model physical parameters:
vp = np.empty(shape, dtype=dtype)
vs = np.empty(shape, dtype=dtype)
rho = np.empty(shape, dtype=dtype)
b = np.empty(shape, dtype=dtype)

vp[:] = 3.5
vs = 0.5 * vp
rho = 0.31*(vp[:]*1000.)**0.25  # Gardner's relation
b = 1./rho  # Buoyancy.

In [ ]:
model =ISOSeismicModel(vp=vp, vs=vs, rho=rho, origin=origin, space_order=space_order, shape=shape, dtype=dtype, spacing=spacing, nbl=nbl)

In [ ]:
plt_options = {'cmap': 'jet', 'extent': [model.origin[0] - nbl * dx,
                    model.origin[0] + model.domain_size[0] + nbl * dx,
                    model.origin[1] + model.domain_size[1] + nbl * dz,
                    model.origin[1] - nbl * dz]}

fig, axes = plt.subplots(3, 3, figsize=(20,13), sharex=True, sharey=True)

img = axes[0,0].imshow(model.vp.data.T, **plt_options, vmin=vs.min(), vmax=vp.max())
axes[0,0].set_title('Vp (km/s)')
axes[0,0].set_ylabel('z (m)')
fig.colorbar(img)
img = axes[0,1].imshow(model.vs.data.T, **plt_options, vmin=vs.min(), vmax=vp.max())
axes[0,1].set_title('Vs (km/s)')
fig.colorbar(img)
img = axes[0,2].imshow(model.rho.data.T, **plt_options)
axes[0,2].set_title('rho (g/cm³)')
fig.colorbar(img)

img = axes[1,0].imshow(model.Ip.data.T, **plt_options, vmin=model.Is.data.min(), vmax=model.Ip.data.max())
axes[1,0].set_title(r'$ Ip \left(\frac{km \cdot g}{s \cdot cm³}\right) $')
axes[1,0].set_ylabel('z (m)')
fig.colorbar(img)
img = axes[1,1].imshow(model.Is.data.T, **plt_options, vmin=model.Is.data.min(), vmax=model.Ip.data.max())
axes[1,1].set_title(r'$ Is \left(\frac{km \cdot g}{s \cdot cm³}\right) $')
fig.colorbar(img)
img = axes[1,2].imshow(model.rho.data.T, **plt_options)
axes[1,2].set_title('rho (g/cm³)')
fig.colorbar(img)

img = axes[2,0].imshow(model.lam.data.T, **plt_options, vmin=model.mu.data.min(), vmax=model.lam.data.max())
axes[2,0].set_title(r'$ \lambda \left(\frac{km^2 \cdot g}{s^2 \cdot cm³}\right) $')
axes[2,0].set_xlabel('x (m)')
axes[2,0].set_ylabel('z (m)')
fig.colorbar(img)
img = axes[2,1].imshow(model.mu.data.T, **plt_options, vmin=model.mu.data.min(), vmax=model.lam.data.max())
axes[2,1].set_title(r'$ \mu \left(\frac{km^2 \cdot g}{s^2 \cdot cm³}\right) $')
axes[2,1].set_xlabel('x (m)')
fig.colorbar(img)
img = axes[2,2].imshow(model.rho.data.T, **plt_options)
axes[2,2].set_title('rho (g/cm³)')
axes[2,2].set_xlabel('x (m)')
fig.colorbar(img)

fig.tight_layout()
plt.show()

In [ ]:
f0 = 0.020  # peak/dominant frequency

s = model.grid.stepping_dim.spacing
damp = model.damp

# Time step in ms and time range:
t0, tn = 0., 1800.
dt = model.critical_dt
time_range = TimeAxis(start=t0, stop=tn, step=dt)

geometry = setup_geometry(model, tn, f0=f0)

In [ ]:
# Function that define source and receiver parameters
def source_rec_term(model, sigma, v0, pos):
    src = RickerSource(name='src', grid=model.grid, f0=f0, time_range=time_range)
    src.coordinates.data[0, :] = pos[0]  # position of source in offset
    src.coordinates.data[0, -1] = pos[1]  # position of source in depth

    src_xx = src.inject(field=sigma[0].forward, expr=src * s)
    src_zz = src.inject(field=sigma[1].forward, expr=src * s)
    src_term = src_xx + src_zz

    # Create symbol for receivers
    rec_vx = Receiver(name='rec_vx', grid=model.grid, npoint=shape[0], time_range=time_range)
    rec_vz = Receiver(name='rec_vz', grid=model.grid, npoint=shape[0], time_range=time_range)
    rec_sigma = Receiver(name='rec_sigma', grid=model.grid, npoint=shape[0], time_range=time_range)

    # Prescribe even spacing for receivers along the x-axis
    rec_vx.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=shape[0])
    rec_vx.coordinates.data[:, 1] = 400.  # postion of receiver at 400 m depth for vx

    rec_vz.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=shape[0])
    rec_vz.coordinates.data[:, 1] = 400.  # postion of receiver at 400 m depth for vz

    rec_sigma.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=shape[0])
    rec_sigma.coordinates.data[:, 1] = 10. # postion of receiver at 10 m of depth for pressure field

    rec_term_vx = rec_vx.interpolate(expr=v0[0])
    rec_term_vz = rec_vz.interpolate(expr=v0[1])
    expr = sigma[0] + sigma[1]
    rec_term_sigma = rec_sigma.interpolate(expr=expr)
    rec_expr = rec_term_vx + rec_term_vz + rec_term_sigma

    return rec_vx, rec_vz, rec_sigma, rec_expr, src_term, src

## Building the forward  operator

The **elastic_forward(...)** function is responsible for forward modeling and makes use of the compact first-order elastic  equation system described below:

\begin{equation}
     \left\{\begin{array}{l}\rho \dfrac{\partial \vec{v}}{\partial t}-D \sigma=0,\\ 
     \dfrac{\partial \sigma}{\partial t}-C D^{T} \vec{v}=f_\sigma, \end{array}\right.
\end{equation}


In [ ]:
# Creating the isotropic elastic tensor
C_vp = C_Matrix(model,'vp-vs-rho')
C_ip = C_Matrix(model,'Ip-Is-rho')
C_lam = C_Matrix(model,'lam-mu')

display(C_vp, C_ip, C_lam)

In [ ]:
def elastic_forward(model, pos, C, **kwargs):

    v0 = VectorTimeFunction(name='v0', grid=model.grid, save=geometry.nt, time_order=1, space_order=space_order)
    sigma = TensorTimeFunction(name='sigma', grid=model.grid, space_order=space_order, save=geometry.nt, time_order=1)
    sigma = vec(sigma)

    pde_v = rho * v0.dt - D(sigma)
    u_v = Eq(v0.forward, damp * solve(pde_v, v0.forward))

    pde_sigma = sigma.dt - C * S(v0.forward)
    u_sigma = Eq(sigma.forward, damp * solve(pde_sigma, sigma.forward))

    rec_vx, rec_vz, rec_sigma, rec_expr, src_term, src = source_rec_term(model, sigma, v0, pos)

    op = Operator([u_v, u_sigma] + src_term + rec_expr, subs=model.spacing_map)
    op(dt=dt, src=src, rec_vx=rec_vx, rec_vz=rec_vz, rec_sigma=rec_sigma, vp=model.vp, vs=model.vs, Ip=model.Ip, Is=model.Is)
    
    return rec_vx, rec_vz, rec_sigma, v0, sigma

In [ ]:
# generating a shot with the source in the center of the model

pos = np.empty((1, 2), dtype=np.float32)
pos[0, 0] = model.domain_size[0] * .5

rec_vx_vp, rec_vz_vp, rec_sigma_vp, v_vp, sigma_vp = elastic_forward(model, pos[0], C_vp)
rec_vx_ip, rec_vz_ip, rec_sigma_ip, v_ip, sigma_ip = elastic_forward(model, pos[0], C_ip)
rec_vx_lam, rec_vz_lam, rec_sigma_lam, v_lam, sigma_lam = elastic_forward(model, pos[0], C_lam)

## Plotting vx, vz and pressure fields shots

In [ ]:
# NBVAL_IGNORE_OUTPUT
slices = [slice(model.nbl, -model.nbl), slice(model.nbl, -model.nbl)]

aspect_ratio = model.shape[0]/model.shape[1]

plt_options_model = {'cmap': 'Greys', 'extent': [model.origin[0], model.origin[0] +
                                                 model.domain_size[0],
                                                 model.origin[1] +
                                                 model.domain_size[1],
                                                 model.origin[1]]}
fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(13, 5))

scale = np.max(rec_vx.data) / 10.

img1 = ax[0].imshow(rec_vx.data, vmin=scale, vmax=-scale, **plt_options_model)
fig.colorbar(img1, ax=ax[0])
ax[0].set_title(r"Shot $v_x$", fontsize=20)
ax[0].set_xlabel('X (m)', fontsize=20)
ax[0].set_ylabel('Time (s)', fontsize=20)
ax[0].set_aspect('auto')

scale2 = np.max(rec_vz.data) / 10.
img2 = ax[1].imshow(rec_vz.data, vmin=scale2, vmax=-scale2, **plt_options_model)
fig.colorbar(img2, ax=ax[1])
ax[1].set_title("Shot $v_z$", fontsize=20)
ax[1].set_xlabel('X (m)', fontsize=20)
ax[1].set_ylabel('Time (s)', fontsize=20)
ax[1].set_aspect('auto')

scale3 = np.max(rec_sigma.data) / 10.
img3 = ax[2].imshow(rec_sigma.data, vmin=scale3, vmax=-scale3, **plt_options_model)
fig.colorbar(img3, ax=ax[2])
ax[2].set_title(r"Shot ($\sigma_{xx}+\sigma_{zz}$)", fontsize=20)
ax[2].set_xlabel('X (m)', fontsize=20)
ax[2].set_ylabel('Time (s)', fontsize=20)
ax[2].set_aspect('auto')

plt.tight_layout()

## Plotting the wavefields snapshots for the particle velocity vector and the stress tensor components

In [ ]:
# NBVAL_IGNORE_OUTPUT
# Some useful definitions for plotting if nbl is set to any other value than zero
nxpad, nzpad = shape[0] + 2 * nbl, shape[1] + 2 * nbl
shape_pad = np.array(shape) + 2 * nbl
origin_pad = tuple([o - s*nbl for o, s in zip(origin, spacing)])
extent_pad = tuple([s*(n-1) for s, n in zip(spacing, shape_pad)])
# Note: flip sense of second dimension to make the plot positive downwards
plt_extent = [origin_pad[0], origin_pad[0] + extent_pad[0],
              origin_pad[1] + extent_pad[1], origin_pad[1]]


def plot(a, title=None):
    # Plot the wavefields, each normalized to scaled maximum of last time step
    kt = (time_range.num - 2) - 1
    amax = 10 * np.max(np.abs(a.data[kt, :, :]))

    nsnaps = 9
    factor = round(time_range.num / nsnaps)

    fig, axes = plt.subplots(1, 4, figsize=(25, 4), sharex=True)
    fig.suptitle(title, size=20)
    for count, ax in enumerate(axes.ravel()):
        snapshot = factor * (count + 1)
        ax.imshow(np.transpose(a.data[snapshot, :, :]), cmap="seismic", vmin=-amax,
                  vmax=+amax, extent=plt_extent)
        ax.plot(model.domain_size[0] * .5, 10, 'red', linestyle='None', marker='*',
                markersize=8, label="Source")
        ax.grid()
        ax.tick_params('both', length=4, width=0.5, which='major', labelsize=10)
        ax.set_title("Wavefield at t=%.2fms" % (factor*count*dt), fontsize=10)
        ax.set_xlabel("X Coordinate (m)", fontsize=10)
        ax.set_ylabel("Z Coordinate (m)", fontsize=10)


plot(v[0], title="Snapshots $v_x$")
plot(v[1], title="Snapshots $v_z$")
plot(sigma[0], title="Snapshots $\sigma_{xx}$")
plot(sigma[1], title="Snapshots $\sigma_{zz}$")
plot(sigma[2], title="Snapshots $\sigma_{xz}$")